In [ ]:
# =============================================================================
# CONTRIBUTOR: Ronak Kamboj
# SECTION: Exploratory Data Analysis (EDA)
# DESCRIPTION: Visualizes target distributions, feature correlations, 
#              and PCA separations using sampled data to prevent memory crashes.
# =============================================================================

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# =============================================================================
# CONFIG
# =============================================================================
INTERIM_PATH = "../data/interim/cleaned_data.csv"
PROCESSED_PATH = "../data/processed/final_data.csv"

SAMPLE_SIZE_HEATMAP = 50000
SAMPLE_SIZE_PCA = 10000

# =============================================================================
# SET STYLE
# =============================================================================
sns.set_theme(style="whitegrid", palette="muted")

# =============================================================================
# LOAD DATA
# =============================================================================
def load_data():
    print("📥 Loading data...")
    
    if not os.path.exists(INTERIM_PATH):
        raise FileNotFoundError(f"Missing file: {INTERIM_PATH}")
    if not os.path.exists(PROCESSED_PATH):
        raise FileNotFoundError(f"Missing file: {PROCESSED_PATH}")

    df_clean = pd.read_csv(INTERIM_PATH)
    df_pca = pd.read_csv(PROCESSED_PATH)

    print("✅ Data loaded successfully!")
    print(f"Cleaned data shape: {df_clean.shape}")
    print(f"PCA data shape: {df_pca.shape}")

    return df_clean, df_pca

# =============================================================================
# CHART 1: TARGET DISTRIBUTION
# =============================================================================
def plot_target_distribution(df):
    print("📊 Plotting target distribution...")

    plt.figure(figsize=(10, 5))
    ax = sns.countplot(data=df, x='label')

    plt.title('Distribution of Target Variable (Label)', fontsize=14, fontweight='bold')
    plt.xlabel('Label Class', fontsize=12)
    plt.ylabel('Count', fontsize=12)

    # Add counts on bars
    for p in ax.patches:
        height = int(p.get_height())
        ax.annotate(
            f'{height}',
            (p.get_x() + p.get_width() / 2., height),
            ha='center',
            va='bottom',
            fontsize=9,
            xytext=(0, 5),
            textcoords='offset points'
        )

    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

# =============================================================================
# CHART 2: CORRELATION HEATMAP
# =============================================================================
def plot_correlation_heatmap(df):
    print("🔥 Generating correlation heatmap...")

    # Sample for performance
    sample_size = min(SAMPLE_SIZE_HEATMAP, len(df))
    df_sample = df.sample(n=sample_size, random_state=42)

    numeric_cols = df_sample.select_dtypes(include='number')

    if numeric_cols.shape[1] == 0:
        print("⚠️ No numeric columns found for correlation.")
        return

    plt.figure(figsize=(12, 8))
    corr_matrix = numeric_cols.corr()

    sns.heatmap(
        corr_matrix,
        annot=False,
        cmap='coolwarm',
        linewidths=0,
        vmin=-1,
        vmax=1
    )

    plt.title('Correlation Heatmap of Features (Sampled)', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

# =============================================================================
# CHART 3: PCA SCATTER
# =============================================================================
def plot_pca_scatter(df):
    print("🧠 Generating PCA scatter plot...")

    required_cols = {'PC1', 'PC2', 'label'}
    if not required_cols.issubset(df.columns):
        print(f"❌ Missing required columns: {required_cols}")
        return

    sample_size = min(SAMPLE_SIZE_PCA, len(df))
    df_sample = df.sample(n=sample_size, random_state=42)

    plt.figure(figsize=(10, 7))

    sns.scatterplot(
        data=df_sample,
        x='PC1',
        y='PC2',
        hue='label',
        palette='viridis',
        alpha=0.7,
        s=60
    )

    plt.title('Scatter Plot: PC1 vs PC2 (Sampled)', fontsize=14, fontweight='bold')
    plt.xlabel('Principal Component 1 (PC1)', fontsize=12)
    plt.ylabel('Principal Component 2 (PC2)', fontsize=12)

    plt.legend(title='Label', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

# =============================================================================
# MAIN
# =============================================================================
def main():
    df_clean, df_pca = load_data()

    plot_target_distribution(df_clean)
    plot_correlation_heatmap(df_clean)
    plot_pca_scatter(df_pca)

# =============================================================================
# ENTRY POINT
# =============================================================================
if __name__ == "__main__":
    main()